In [0]:

print("ICICI pipeline setup")

raw_path = "/Volumes/dbacademy/default/icici_raw"

import os
print("Volume accessible:", os.path.exists(raw_path))

ICICI pipeline setup
Volume accessible: True


In [0]:
# Test connection
import requests

BASE_URL = "https://zucau19vp0.execute-api.us-east-1.amazonaws.com"

response = requests.get(
    f"{BASE_URL}/files",
    params={"project": "icici", "path": "2025/01/01/"},
    timeout=30
)

print("HTTP status:", response.status_code)
print("Response:", response.text[:2000])

HTTP status: 200
Response: {"bucket": "training-2027-raw-data", "project": "icici", "path": "2025/01/01/", "files": [{"key": "icici/2025/01/01/account.csv", "size": 140065, "last_modified": "2026-10-07T15:21:09+00:00"}, {"key": "icici/2025/01/01/account.csv.gz", "size": 34472, "last_modified": "2026-10-07T15:21:02+00:00"}, {"key": "icici/2025/01/01/account_holder.csv", "size": 62963, "last_modified": "2026-10-07T15:21:14+00:00"}, {"key": "icici/2025/01/01/account_holder.csv.gz", "size": 8450, "last_modified": "2026-10-07T15:21:12+00:00"}, {"key": "icici/2025/01/01/card.csv", "size": 136722, "last_modified": "2026-10-07T15:21:13+00:00"}, {"key": "icici/2025/01/01/card.csv.gz", "size": 32699, "last_modified": "2026-10-07T15:20:58+00:00"}, {"key": "icici/2025/01/01/card_transaction.csv", "size": 315336, "last_modified": "2026-10-07T15:21:04+00:00"}, {"key": "icici/2025/01/01/card_transaction.csv.gz", "size": 55708, "last_modified": "2026-10-07T15:21:07+00:00"}, {"key": "icici/2025/01/01/c

In [0]:


import requests
import os
from concurrent.futures import ThreadPoolExecutor

BASE_URL = "https://zucau19vp0.execute-api.us-east-1.amazonaws.com"
RAW_PATH = "/Volumes/dbacademy/default/icici_raw"

# 1. Fetch file listings from API
files = []

for month in range(1, 13):
    path = f"2025/{month:02d}/"
    response = requests.get(
        f"{BASE_URL}/files",
        params={"project": "icici", "path": path},
        timeout=30
    )
    response.raise_for_status()
    for item in response.json().get("files", []):
        key = item["key"]
        if key.lower().endswith(".csv"):
            parts = key.split("/")
            files.append({
                "key": key,
                "month": parts[2],
                "day": parts[3],
                "filename": parts[4],
            })

print(f"Total CSV files to download: {len(files)}")



def download(item):
    folder = os.path.join(RAW_PATH, "2025", item["month"], item["day"])
    os.makedirs(folder, exist_ok=True)
    local_file = os.path.join(folder, item["filename"])
    url_response = requests.get(
        f"{BASE_URL}/download",
        params={"project": "icici", "path": item["key"].split("icici/", 1)[-1]},
        timeout=30,
    )
    url_response.raise_for_status()
    download_url = url_response.json()["download_url"]
    with requests.get(download_url, stream=True, timeout=(10, 120)) as r:
        r.raise_for_status()
        with open(local_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    return local_file


with ThreadPoolExecutor(max_workers=10) as pool:
    results = list(pool.map(download, files))

print(f"Downloaded: {len(results)} files")


Total CSV files: 2880
Downloaded files: 2880
account: 180000 rows, 22 columns
account_holder: 180000 rows, 11 columns
card: 144000 rows, 23 columns
card_transaction: 300000 rows, 24 columns
customer: 120000 rows, 32 columns
customer_address: 120000 rows, 16 columns
deposit: 120000 rows, 18 columns
insurance_policy: 120000 rows, 19 columns
insurance_premium: 144000 rows, 15 columns
investment_account: 120000 rows, 13 columns
investment_holding: 240000 rows, 16 columns
kyc: 120000 rows, 17 columns
loan: 120000 rows, 25 columns
loan_payment: 180000 rows, 19 columns
transaction: 600000 rows, 29 columns
wealth_profile: 120000 rows, 16 columns


account_id,customer_id,account_type,product_code,currency,branch_id,open_date,close_date,status,balance,available_balance,interest_rate,min_balance,lien_amount,nominee_flag,mode_of_operation,relationship_manager_id,source_system,source_file,year,month,day
ACC000091001,CUST000034150,CASA,SA001,INR,BR0103,2025-07-03,null,DORMANT,784103.23,779640.28,5.614,1000,2040.56,true,Singly,RM02157,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091002,CUST000053464,CASA,SA001,INR,BR0253,2025-07-03,null,ACTIVE,813037.04,810642.42,5.663,5000,1097.58,true,Former or Survivor,RM07554,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091003,CUST000009342,CASA,SA002,INR,BR0328,2025-07-03,null,ACTIVE,884540.05,881153.6000000001,4.875,10000,4459.3,false,Either or Survivor,RM02866,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091004,CUST000055034,CASA,SA002,INR,BR0217,2025-07-03,null,ACTIVE,674462.17,674310.7300000001,3.442,1000,751.1,false,Jointly,RM07581,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091005,CUST000004071,CASA,CA001,INR,BR0323,2025-07-03,null,ACTIVE,736761.59,735728.0399999999,3.37,5000,2859.42,false,Former or Survivor,RM02534,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091006,CUST000051289,CASA,SA002,INR,BR0029,2025-07-03,null,ACTIVE,464611.99,460708.64,4.845,5000,2863.1,true,Jointly,RM03643,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091007,CUST000015507,CASA,CA001,INR,BR0064,2025-07-03,null,ACTIVE,632133.64,629252.92,5.287,0,1441.97,true,Former or Survivor,RM09975,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091008,CUST000057276,CASA,CA001,INR,BR0348,2025-07-03,null,DORMANT,625667.06,621628.4500000001,5.505,10000,2540.44,false,Either or Survivor,RM06257,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091009,CUST000036176,CASA,SA001,INR,BR0227,2025-07-03,null,DORMANT,812523.82,809372.6799999999,3.185,1000,62.59,true,Former or Survivor,RM00606,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03
ACC000091010,CUST000059382,CASA,SA002,INR,BR0280,2025-07-03,null,DORMANT,299587.89,299186.5,6.9,5000,170.2,true,Former or Survivor,RM02709,CBS,dbfs:/Volumes/dbacademy/default/icici_raw/2025/07/03/account.csv,2025,07,03


In [0]:
# STAGING LAYER: Load CSVs to Delta Staging Tables

import os
from pyspark.sql.functions import col, current_timestamp

RAW_PATH = "/Volumes/dbacademy/default/icici_raw"
STAGING_SCHEMA = "dbacademy.icici_staging"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {STAGING_SCHEMA}")

# Discover table names from downloaded CSVs in the volume
table_names = sorted(set(
    fname[:-4]
    for _, _, file_list in os.walk(RAW_PATH)
    for fname in file_list
    if fname.lower().endswith(".csv")
))

for table_name in table_names:
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{RAW_PATH}/2025/*/*/{table_name}.csv")
        .withColumn("_staging_file_path", col("_metadata.file_path"))
        .withColumn("_staging_load_ts", current_timestamp())
    )
    row_count = df.count()
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{STAGING_SCHEMA}.{table_name}")
    )
    print(f"  {table_name}: {row_count:,} rows loaded to staging")

print(f"\n✓ Staging layer complete: {len(table_names)} tables in {STAGING_SCHEMA}")

  account: 180,000 rows loaded to staging
  account_holder: 180,000 rows loaded to staging
  card: 144,000 rows loaded to staging
  card_transaction: 300,000 rows loaded to staging
  customer: 120,000 rows loaded to staging
  customer_address: 120,000 rows loaded to staging
  deposit: 120,000 rows loaded to staging
  insurance_policy: 120,000 rows loaded to staging
  insurance_premium: 144,000 rows loaded to staging
  investment_account: 120,000 rows loaded to staging
  investment_holding: 240,000 rows loaded to staging
  kyc: 120,000 rows loaded to staging
  loan: 120,000 rows loaded to staging
  loan_payment: 180,000 rows loaded to staging
  transaction: 600,000 rows loaded to staging
  wealth_profile: 120,000 rows loaded to staging

✓ Staging layer complete: 16 tables in dbacademy.icici_staging


In [0]:
# BRONZE LAYER: Staging Delta → Bronze Delta with Metadata

from pyspark.sql.functions import col, regexp_extract, current_timestamp

BRONZE_SCHEMA = "dbacademy.icici_bronze"
STAGING_SCHEMA = "dbacademy.icici_staging"

# Discover table names from staging schema
table_names = [
    r.tableName
    for r in spark.sql(f"SHOW TABLES IN {STAGING_SCHEMA}").collect()
]

for table_name in table_names:
    df = (
        spark.table(f"{STAGING_SCHEMA}.{table_name}")
        .withColumns({
            "_source_file": col("_staging_file_path"),
            "_load_ts": current_timestamp(),
            "year": regexp_extract(col("_staging_file_path"), r"/(\d{4})/", 1),
            "month": regexp_extract(col("_staging_file_path"), r"/(\d{4})/(\d{2})/", 2),
            "day": regexp_extract(col("_staging_file_path"), r"/(\d{4})/(\d{2})/(\d{2})/", 3),
        })
        .drop("_staging_file_path", "_staging_load_ts")
    )
    row_count = df.count()
    (
        df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    )
    print(f"  {table_name}: {row_count:,} rows")

print(f"\n✓ Bronze layer complete: {len(table_names)} tables in {BRONZE_SCHEMA}")

In [0]:
# CDC: Transaction Log-Based Change Tracking (Staging & Bronze)

from pyspark.sql.functions import col, current_timestamp

CDC_SCHEMA = "dbacademy.icici_cdc"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CDC_SCHEMA}")

# 1. Enable Delta Change Data Feed on staging and bronze tables
staging_tables = [r.tableName for r in spark.sql("SHOW TABLES IN dbacademy.icici_staging").collect()]
bronze_tables = [r.tableName for r in spark.sql("SHOW TABLES IN dbacademy.icici_bronze").collect()]

for t in staging_tables:
    spark.sql(f"ALTER TABLE dbacademy.icici_staging.{t} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
for t in bronze_tables:
    spark.sql(f"ALTER TABLE dbacademy.icici_bronze.{t} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
print(f"✓ CDF enabled: {len(staging_tables)} staging + {len(bronze_tables)} bronze tables")

# 2. CDC version tracker — last processed Delta version per table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CDC_SCHEMA}._cdc_version_tracker (
    layer STRING, table_name STRING, last_version LONG, last_cdc_ts TIMESTAMP
) USING DELTA
""")

# 3. CDC change log — audit trail of all detected changes
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CDC_SCHEMA}.change_log (
    layer STRING, table_name STRING, change_type STRING,
    record_count LONG, from_version LONG, to_version LONG, captured_ts TIMESTAMP
) USING DELTA
""")

# 4. Helper: get latest Delta version via transaction log (DESCRIBE HISTORY)
def get_latest_version(table_fqn):
    hist = spark.sql(f"DESCRIBE HISTORY {table_fqn}").collect()
    return max(r["version"] for r in hist) if hist else 0

# 5. Initialize version tracker to current versions (baseline)
for layer, tables in [("staging", staging_tables), ("bronze", bronze_tables)]:
    for t in tables:
        fqn = f"dbacademy.icici_{layer}.{t}"
        ver = get_latest_version(fqn)
        spark.sql(f"""
        MERGE INTO {CDC_SCHEMA}._cdc_version_tracker AS tgt
        USING (SELECT '{layer}' AS layer, '{t}' AS table_name, {ver} AS last_version, current_timestamp() AS last_cdc_ts) AS src
        ON tgt.layer = src.layer AND tgt.table_name = src.table_name
        WHEN MATCHED THEN UPDATE SET last_version = src.last_version, last_cdc_ts = src.last_cdc_ts
        WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"  {layer}.{t} → baseline version {ver}")

# 6. capture_changes() — detect and log changes since last tracked version
def capture_changes(layer, table_name):
    fqn = f"dbacademy.icici_{layer}.{table_name}"
    row = spark.sql(f"SELECT last_version FROM {CDC_SCHEMA}._cdc_version_tracker WHERE layer = '{layer}' AND table_name = '{table_name}'").collect()
    start_ver = row[0][0] if row else 0
    latest_ver = get_latest_version(fqn)

    if start_ver >= latest_ver:
        print(f"  {layer}.{table_name}: no changes since v{start_ver}")
        return []

    changes = spark.sql(f"SELECT * FROM table_changes('{fqn}', {start_ver + 1})")
    counts = changes.groupBy("_change_type").count().collect()

    for c in counts:
        spark.sql(f"""
        INSERT INTO {CDC_SCHEMA}.change_log
        VALUES ('{layer}', '{table_name}', '{c['_change_type']}', {c['count']}, {start_ver}, {latest_ver}, current_timestamp())
        """)

    # Advance version tracker
    spark.sql(f"""
    MERGE INTO {CDC_SCHEMA}._cdc_version_tracker AS tgt
    USING (SELECT '{layer}' AS layer, '{table_name}' AS table_name, {latest_ver} AS last_version, current_timestamp() AS last_cdc_ts) AS src
    ON tgt.layer = src.layer AND tgt.table_name = src.table_name
    WHEN MATCHED THEN UPDATE SET last_version = src.last_version, last_cdc_ts = src.last_cdc_ts
    """)

    for c in counts:
        print(f"  {layer}.{table_name}: {c['_change_type']} = {c['count']:,}")
    return [(c['_change_type'], c['count']) for c in counts]

# 7. incremental_bronze_load() — MERGE staging into bronze for subsequent loads

primary_keys = {
    "account": "account_id", "account_holder": "account_holder_id",
    "card": "card_id", "card_transaction": "card_txn_id",
    "customer": "customer_id", "customer_address": "address_id",
    "deposit": "deposit_id", "insurance_policy": "policy_id",
    "insurance_premium": "premium_id", "investment_account": "investment_account_id",
    "investment_holding": "holding_id", "kyc": "kyc_id",
    "loan": "loan_id", "loan_payment": "payment_id",
    "transaction": "transaction_id", "wealth_profile": "wealth_profile_id",
}

def incremental_bronze_load(table_name):
    pk = primary_keys[table_name]
    staging_df = spark.table(f"dbacademy.icici_staging.{table_name}")
    staging_df.createOrReplaceTempView(f"_stg_{table_name}")
    spark.sql(rf"""
    MERGE INTO dbacademy.icici_bronze.{table_name} AS tgt
    USING (
        SELECT
            *,
            regexp_extract(_staging_file_path, r'/(\d{{4}})/', 1) AS year,
            regexp_extract(_staging_file_path, r'/(\d{{4}})/(\d{{2}})/', 2) AS month,
            regexp_extract(_staging_file_path, r'/(\d{{4}})/(\d{{2}})/(\d{{2}})/', 3) AS day,
            _staging_file_path AS _source_file,
            current_timestamp() AS _load_ts
        FROM _stg_{table_name}
    ) AS src
    ON tgt.{pk} = src.{pk}
    WHEN MATCHED AND tgt._load_ts < src._load_ts THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"  {table_name}: MERGE complete (pk={pk})")
    return capture_changes("bronze", table_name)

# 8. Show transaction log history 
print("\n--- Delta Transaction Log: bronze.transaction ---")
display(
    spark.sql("DESCRIBE HISTORY dbacademy.icici_bronze.transaction")
    .select("version", "operation", "operationMetrics")
    .limit(5)
)

print(f"\n✓ CDC infrastructure ready in {CDC_SCHEMA}")
print("  • Baseline versions recorded via DESCRIBE HISTORY")
print("  • Call capture_changes('bronze', 'table_name') after subsequent loads")
print("  • Call incremental_bronze_load('table_name') for MERGE-based reloads")

✓ CDF enabled: 16 staging + 16 bronze tables
  staging.account → baseline version 4
  staging.account_holder → baseline version 4
  staging.card → baseline version 4
  staging.card_transaction → baseline version 4
  staging.customer → baseline version 4
  staging.customer_address → baseline version 4
  staging.deposit → baseline version 4
  staging.insurance_policy → baseline version 4
  staging.insurance_premium → baseline version 4
  staging.investment_account → baseline version 4
  staging.investment_holding → baseline version 4
  staging.kyc → baseline version 4
  staging.loan → baseline version 4
  staging.loan_payment → baseline version 4
  staging.transaction → baseline version 4
  staging.wealth_profile → baseline version 4
  bronze.account → baseline version 21
  bronze.account_holder → baseline version 21
  bronze.card → baseline version 21
  bronze.card_transaction → baseline version 21
  bronze.customer → baseline version 21
  bronze.customer_address → baseline version 21
 

version,operation,operationMetrics
21,SET TBLPROPERTIES,Map()
20,SET TBLPROPERTIES,Map()
19,SET TBLPROPERTIES,Map()
18,SET TBLPROPERTIES,Map()
17,SET TBLPROPERTIES,Map()



✓ CDC infrastructure ready in dbacademy.icici_cdc
  • Baseline versions recorded via DESCRIBE HISTORY
  • Call capture_changes('bronze', 'table_name') after subsequent loads
  • Call incremental_bronze_load('table_name') for MERGE-based reloads


In [0]:
# PII VAULT + SILVER LAYER: Surrogate Keys, 3NF Normalization
from pyspark.sql.functions import (
    col, to_date, when, trim, row_number, xxhash64
)
from pyspark.sql.window import Window

# Create PII Vault schema (access-controlled)
spark.sql("CREATE SCHEMA IF NOT EXISTS dbacademy.icici_pii")

# Customer PII: names, DOB, masked IDs → stored with surrogate key
_customer_pii = (
    spark.table("dbacademy.icici_bronze.customer")
    .select(
        "customer_id", "first_name", "last_name", "dob",
        "pan_masked", "ckyc_id_masked", "mobile_masked", "email_masked",
    )
    .dropDuplicates(["customer_id"])
    .withColumn("customer_pii_key", xxhash64(col("customer_id")))
)
(_customer_pii.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dbacademy.icici_pii.customer_pii_map"))

# Address PII: line1, line2 → stored securely
_address_pii = (
    spark.table("dbacademy.icici_bronze.customer_address")
    .select("address_id", "line1", "line2")
    .dropDuplicates(["address_id"])
    .withColumn("address_pii_key", xxhash64(col("address_id")))
)
(_address_pii.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dbacademy.icici_pii.address_pii_map"))

# Transaction PII: counterparty_name, narration → stored securely
_txn_pii = (
    spark.table("dbacademy.icici_bronze.transaction")
    .select("transaction_id", "counterparty_name", "narration")
    .dropDuplicates(["transaction_id"])
    .withColumn("counterparty_pii_key", xxhash64(col("transaction_id")))
)
(_txn_pii.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("dbacademy.icici_pii.transaction_pii_map"))

print("✓ PII vault: customer_pii_map, address_pii_map, transaction_pii_map")

# Silver Layer: 3NF tables with surrogate keys (no PII) 
silver_schema = "dbacademy.icici_silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

primary_keys = {
    "account": "account_id", "account_holder": "account_holder_id",
    "card": "card_id", "card_transaction": "card_txn_id",
    "customer": "customer_id", "customer_address": "address_id",
    "deposit": "deposit_id", "insurance_policy": "policy_id",
    "insurance_premium": "premium_id", "investment_account": "investment_account_id",
    "investment_holding": "holding_id", "kyc": "kyc_id",
    "loan": "loan_id", "loan_payment": "payment_id",
    "transaction": "transaction_id", "wealth_profile": "wealth_profile_id",
}

date_casts = {
    "account": {"close_date": "close_date"},
    "account_holder": {"end_date": "end_date"},
    "customer_address": {"valid_to": "valid_to"},
}

quality_checks = {
    "account": ["(balance >= 0 OR balance IS NULL)"],
    "transaction": ["(amount > 0 OR amount IS NULL)"],
    "loan": ["(sanctioned_amount >= 0 OR sanctioned_amount IS NULL)"],
    "deposit": ["(principal_amount > 0 OR principal_amount IS NULL)"],
    "card": ["(credit_limit >= 0 OR credit_limit IS NULL)"],
}

# PII: (vault_table, join_col, surrogate_key_col, [columns_to_drop])
pii_joins = {
    "customer": ("dbacademy.icici_pii.customer_pii_map", "customer_id", "customer_pii_key",
                 ["first_name", "last_name", "dob", "pan_masked", "ckyc_id_masked", "mobile_masked", "email_masked"]),
    "customer_address": ("dbacademy.icici_pii.address_pii_map", "address_id", "address_pii_key",
                         ["line1", "line2"]),
    "transaction": ("dbacademy.icici_pii.transaction_pii_map", "transaction_id", "counterparty_pii_key",
                    ["counterparty_name", "narration"]),
}

for table_name in sorted(primary_keys.keys()):
    df = spark.table(f"dbacademy.icici_bronze.{table_name}")
    pk = primary_keys[table_name]

    # Deduplicate by primary key, keep latest record
    w = Window.partitionBy(pk).orderBy(col("_load_ts").desc())
    df = (
        df.withColumns({"_rn": row_number().over(w)})
        .filter(col("_rn") == 1)
        .drop("_rn")
    )

    # Cast string date columns to proper date type
    if table_name in date_casts:
        cast_exprs = {}
        for src_col, tgt_col in date_casts[table_name].items():
            cast_exprs[tgt_col] = when(trim(col(src_col)) == "", None).otherwise(to_date(col(src_col)))
        df = df.withColumns(cast_exprs)

    # Apply data quality filters
    for condition in quality_checks.get(table_name, []):
        df = df.filter(condition)

    # Replace PII with surrogate keys, drop original PII columns
    if table_name in pii_joins:
        pii_table, join_col, pii_key_col, pii_drop_cols = pii_joins[table_name]
        pii_df = spark.table(pii_table).select(join_col, pii_key_col)
        df = df.join(pii_df, join_col, "left")
        for c in pii_drop_cols:
            df = df.drop(c)

    row_count = df.count()
    (
        df.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{silver_schema}.{table_name}")
    )
    pii_tag = " [PII → surrogate keys]" if table_name in pii_joins else ""
    print(f"  {table_name}: {row_count:,} rows (3NF{pii_tag})")

print(f"\n✓ Silver layer complete: {len(primary_keys)} 3NF tables in {silver_schema}")
print("✓ PII vault in dbacademy.icici_pii (access-controlled)")

✓ PII vault: customer_pii_map, address_pii_map, transaction_pii_map
  account: 180,000 rows (3NF)
  account_holder: 180,000 rows (3NF)
  card: 144,000 rows (3NF)
  card_transaction: 300,000 rows (3NF)
  customer: 120,000 rows (3NF [PII → surrogate keys])
  customer_address: 120,000 rows (3NF [PII → surrogate keys])
  deposit: 120,000 rows (3NF)
  insurance_policy: 120,000 rows (3NF)
  insurance_premium: 144,000 rows (3NF)
  investment_account: 120,000 rows (3NF)
  investment_holding: 240,000 rows (3NF)
  kyc: 120,000 rows (3NF)
  loan: 120,000 rows (3NF)
  loan_payment: 180,000 rows (3NF)
  transaction: 600,000 rows (3NF [PII → surrogate keys])
  wealth_profile: 120,000 rows (3NF)

✓ Silver layer complete: 16 3NF tables in dbacademy.icici_silver
✓ PII vault in dbacademy.icici_pii (access-controlled)


In [0]:
# GOLD LAYER: Star Schema with MERGE Operations

gold_schema = "dbacademy.icici_gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# Dimension Tables 

# dim_customer
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_customer AS
SELECT
    xxhash64(c.customer_id) AS customer_key,
    c.customer_id, c.customer_pii_key,
    c.customer_type, c.gender, c.segment, c.risk_category,
    c.kyc_status, c.annual_income, c.occupation, c.employment_type,
    c.employer_name, c.resident_country, c.nationality, c.marital_status,
    c.source_of_funds, c.preferred_language, c.preferred_channel,
    c.consent_marketing, c.created_ts, c.updated_ts, c.record_status
FROM dbacademy.icici_silver.customer c
""")
print(f"  dim_customer: {spark.table(f'{gold_schema}.dim_customer').count():,} rows")

# dim_account
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_account AS
SELECT
    xxhash64(a.account_id) AS account_key,
    a.account_id, xxhash64(a.customer_id) AS customer_key,
    a.account_type, a.product_code, a.currency, a.branch_id,
    a.status, a.open_date, a.close_date, a.interest_rate,
    a.min_balance, a.nominee_flag, a.mode_of_operation,
    a.relationship_manager_id, a.source_system
FROM dbacademy.icici_silver.account a
""")
print(f"  dim_account: {spark.table(f'{gold_schema}.dim_account').count():,} rows")

# dim_card
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_card AS
SELECT
    xxhash64(c.card_id) AS card_key,
    c.card_id, xxhash64(c.customer_id) AS customer_key,
    xxhash64(c.account_id) AS account_key,
    c.card_type, c.card_network, c.product_name, c.status,
    c.issue_date, c.activation_date, c.expiry_date,
    c.billing_cycle_day, c.international_enabled,
    c.contactless_enabled, c.tokenization_status
FROM dbacademy.icici_silver.card c
""")
print(f"  dim_card: {spark.table(f'{gold_schema}.dim_card').count():,} rows")

# dim_date (built from all date columns across fact tables)
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_date AS
SELECT DISTINCT
    d.date_val AS date_key,
    YEAR(d.date_val) AS year, MONTH(d.date_val) AS month,
    DAY(d.date_val) AS day, QUARTER(d.date_val) AS quarter,
    DATE_FORMAT(d.date_val, 'MMMM') AS month_name,
    DAYOFWEEK(d.date_val) AS day_of_week,
    DAYOFYEAR(d.date_val) AS day_of_year
FROM (
    SELECT DISTINCT open_date AS date_val FROM dbacademy.icici_silver.account WHERE open_date IS NOT NULL
    UNION
    SELECT DISTINCT CAST(txn_ts AS DATE) AS date_val FROM dbacademy.icici_silver.transaction WHERE txn_ts IS NOT NULL
    UNION
    SELECT DISTINCT application_date AS date_val FROM dbacademy.icici_silver.loan WHERE application_date IS NOT NULL
) d
""")
print(f"  dim_date: {spark.table(f'{gold_schema}.dim_date').count():,} rows")

# dim_branch (derived from accounts)
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_branch AS
SELECT
    branch_id,
    COUNT(DISTINCT account_id) AS account_count,
    COUNT(DISTINCT customer_id) AS customer_count
FROM dbacademy.icici_silver.account
GROUP BY branch_id
""")
print(f"  dim_branch: {spark.table(f'{gold_schema}.dim_branch').count():,} rows")

# dim_loan
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.dim_loan AS
SELECT
    xxhash64(l.loan_id) AS loan_key,
    l.loan_id, xxhash64(l.customer_id) AS customer_key,
    l.loan_type, l.product_code, l.application_date, l.sanction_date,
    l.disbursement_date, l.sanctioned_amount, l.disbursed_amount,
    l.interest_rate, l.tenure_months, l.emi_amount,
    l.repayment_frequency, l.collateral_type, l.ltv_pct,
    l.npa_stage, l.days_past_due, l.status, l.branch_id, l.sourcing_channel
FROM dbacademy.icici_silver.loan l
""")
print(f"  dim_loan: {spark.table(f'{gold_schema}.dim_loan').count():,} rows")

# Fact Tables

# fact_transactions
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.fact_transactions AS
SELECT
    xxhash64(t.transaction_id) AS transaction_key,
    xxhash64(t.customer_id) AS customer_key,
    xxhash64(t.account_id) AS account_key,
    CAST(t.txn_ts AS DATE) AS date_key,
    t.transaction_id, t.txn_type, t.debit_credit, t.amount,
    t.currency, t.balance_after, t.channel, t.payment_mode,
    t.merchant_id, t.merchant_category, t.counterparty_pii_key,
    t.txn_status, t.reversal_flag, t.fraud_flag, t.source_system
FROM dbacademy.icici_silver.transaction t
""")
print(f"  fact_transactions: {spark.table(f'{gold_schema}.fact_transactions').count():,} rows")

# fact_card_transactions
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.fact_card_transactions AS
SELECT
    xxhash64(ct.card_txn_id) AS card_txn_key,
    xxhash64(ct.card_id) AS card_key,
    xxhash64(ct.customer_id) AS customer_key,
    CAST(ct.txn_ts AS DATE) AS date_key,
    ct.card_txn_id, ct.txn_type, ct.amount, ct.currency,
    ct.merchant_city, ct.merchant_country, ct.mcc, ct.channel,
    ct.emi_flag, ct.emi_tenure, ct.txn_status,
    ct.fraud_flag, ct.dispute_flag, ct.settlement_date
FROM dbacademy.icici_silver.card_transaction ct
""")
print(f"  fact_card_transactions: {spark.table(f'{gold_schema}.fact_card_transactions').count():,} rows")

# fact_loan_payments
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.fact_loan_payments AS
SELECT
    xxhash64(lp.payment_id) AS payment_key,
    xxhash64(lp.loan_id) AS loan_key,
    xxhash64(lp.customer_id) AS customer_key,
    lp.payment_date AS date_key,
    lp.payment_id, lp.due_date, lp.scheduled_amount,
    lp.principal_component, lp.interest_component,
    lp.penalty_amount, lp.payment_amount, lp.payment_channel,
    lp.payment_status, lp.days_past_due, lp.dpd_bucket, lp.receipt_ref
FROM dbacademy.icici_silver.loan_payment lp
""")
print(f"  fact_loan_payments: {spark.table(f'{gold_schema}.fact_loan_payments').count():,} rows")

# fact_account_balances (current snapshot)
spark.sql(f"""
CREATE OR REPLACE TABLE {gold_schema}.fact_account_balances AS
SELECT
    xxhash64(a.account_id) AS account_key,
    xxhash64(a.customer_id) AS customer_key,
    a.balance, a.available_balance, a.interest_rate,
    a.min_balance, a.lien_amount
FROM dbacademy.icici_silver.account a
""")
print(f"  fact_account_balances: {spark.table(f'{gold_schema}.fact_account_balances').count():,} rows")

print(f"\n✓ Gold star schema complete: 6 dimensions + 4 fact tables in {gold_schema}")

# Verify star schema with a sample join 
print("\n--- Sample: Transaction with dimensions ---")
display(spark.sql(f"""
SELECT f.amount, f.txn_type, f.debit_credit,
       dc.segment AS customer_segment, dc.risk_category,
       da.account_type, da.status AS account_status,
       dd.month_name, dd.year
FROM {gold_schema}.fact_transactions f
JOIN {gold_schema}.dim_customer dc ON f.customer_key = dc.customer_key
JOIN {gold_schema}.dim_account da ON f.account_key = da.account_key
JOIN {gold_schema}.dim_date dd ON f.date_key = dd.date_key
LIMIT 15
"""))

  dim_customer: 120,000 rows
  dim_account: 180,000 rows
  dim_card: 144,000 rows
  dim_date: 423 rows
  dim_branch: 400 rows
  dim_loan: 120,000 rows
  fact_transactions: 600,000 rows
  fact_card_transactions: 300,000 rows
  fact_loan_payments: 180,000 rows
  fact_account_balances: 180,000 rows

✓ Gold star schema complete: 6 dimensions + 4 fact tables in dbacademy.icici_gold

--- Sample: Transaction with dimensions ---


amount,txn_type,debit_credit,customer_segment,risk_category,account_type,account_status,month_name,year
25547.73,Purchase,C,Emerging Affluent,High,CASA,DORMANT,January,2025
222581.66,Cash Deposit,C,Mass Retail,Medium,CASA,DORMANT,January,2025
106315.62,Refund,C,Affluent,Medium,CASA,ACTIVE,January,2025
178321.32,Bill Payment,D,Emerging Affluent,High,CASA,DORMANT,January,2025
167917.94,Salary Credit,D,Emerging Affluent,Medium,CASA,DORMANT,January,2025
123831.24,Salary Credit,D,Private Banking,High,CASA,ACTIVE,January,2025
40673.81,Cash Deposit,C,Emerging Affluent,Low,CASA,ACTIVE,January,2025
225086.31,Cash Withdrawal,D,Mass Retail,High,CASA,ACTIVE,January,2025
80746.76,EMI,D,Private Banking,Low,CASA,ACTIVE,January,2025
230767.27,Bill Payment,C,NRI,Medium,CASA,ACTIVE,January,2025
